# ⏱️ Chapter 07: 15-Minute City Walking Isochrones & Healthcare Deserts
## Network Reachability Analysis Using pgRouting and Concave Hulls

### 🎯 The Mission: Why We Are Doing This
The **15-Minute City concept** mandates that residents should reach primary healthcare within a 15-minute walk ($1.125	ext{ km}$ at $1.25	ext{ m/s}$). We compute driving-distance trees with `pgr_drivingDistance` and wrap reachable nodes into isochrone polygons to detect healthcare deserts.

---

### 🔑 Spatial DBA Keywords Decoded

| Function | Plain-English Explanation | Production Note |
| :--- | :--- | :--- |
| **`pgr_drivingDistance`** | Executes Dijkstra graph search outward up to travel-time threshold $T_{\max}$. | Returns all reachable road network nodes. |
| **`ST_ConcaveHull(pts, target)`** | Wraps point clouds into tight, concave boundaries without excessive convex ballooning. | Creates realistic polygon reachability contours. |
| **Service Desert** | City areas outside the reachability polygon. | Critical for planning new hospital investments. |

In [ ]:
from config_db import execute_sql, query_df, query_gdf, preview_map

sql_iso = """
DROP TABLE IF EXISTS health_origins CASCADE;
CREATE TABLE health_origins AS
SELECT 
    landmark_id, name, geom_utm30n,
    (SELECT v.id FROM accra_streets_vertices_pgr v ORDER BY v.the_geom <-> l.geom_utm30n LIMIT 1) AS start_vertex
FROM accra_landmarks l
WHERE category = 'Healthcare';

DROP TABLE IF EXISTS clinic_15min_isochrone CASCADE;
CREATE TABLE clinic_15min_isochrone AS
WITH reachable_nodes AS (
    SELECT 
        h.landmark_id, h.name, dd.node AS vertex_id, v.the_geom AS geom_vertex
    FROM health_origins h
    CROSS JOIN LATERAL pgr_drivingDistance(
        'SELECT id, source, target, cost_len / 1.25 AS cost, cost_len / 1.25 AS reverse_cost FROM accra_streets',
        h.start_vertex,
        900.0,
        directed := false
    ) dd
    JOIN accra_streets_vertices_pgr v ON dd.node = v.id
)
SELECT 
    landmark_id, name,
    ST_ConcaveHull(ST_Collect(geom_vertex), 0.75)::GEOMETRY(Polygon, 32630) AS geom
FROM reachable_nodes
GROUP BY landmark_id, name;

CREATE INDEX idx_isochrone_geom ON clinic_15min_isochrone USING GIST(geom);
"""

execute_sql(sql_iso)
print("✅ 15-Minute walking isochrone polygons generated.")

In [ ]:
# View Isochrone Area Metrics
df_iso = query_df("""
SELECT landmark_id, name, ROUND((ST_Area(geom) / 10000.0)::NUMERIC, 2) AS coverage_hectares
FROM clinic_15min_isochrone;
""")

df_iso

In [ ]:
# Interactive Map Preview of Isochrone Catchments
gdf_iso = query_gdf("""
SELECT landmark_id, name, geom FROM clinic_15min_isochrone;
""")

preview_map(gdf_iso, tooltip_cols=["name"])